# Maintenance-fee lapse review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** B2/B3 — dark-majority follow-up: weak-negative liveness evidence  
([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/nano_dark_firm_maintenance_lapses.py`  
**Data as of:** USPTO maintenance-fee events file `MaintFeeEvents_20260707`  

Companion view over the lapse artifact, focused on event timing and on how sensitive
the firm-level dormancy flag is to its threshold definition. Exploratory-tier and
non-citable.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

## Data contract

- **Population:** the high-confidence patent-matched dark firms from
  `nano_dark_firm_liveness.py` — identity matches were reused, not re-derived.
- **Grain:** firm (per-patent events are aggregated inside the script).
- **Keys:** `normalized_name`; `company` is the display label.
- **Signal design:** lapse = EXP. event with no later EXPX reinstatement. Only patents
  old enough to have crossed their first 3.5-year fee checkpoint contribute signal.
  The canonical dormancy flag is ≥80% of checkpoint-eligible patents lapsed AND ≥2
  eligible patents.
- **Caveats:** this is the instrument set's first WEAK NEGATIVE signal. A full-portfolio
  lapse is suggestive of disengagement, not proof of dissolution — deliberate patent
  abandonment by a healthy, pivoted, or trade-secret firm looks identical here.

In [ ]:
ARTIFACTS = {
    "maintenance lapses": REPORT_DIR / "dark_firm_maintenance_lapses.csv",
}
GENERATORS = {
    "maintenance lapses": "scripts/data/nano_dark_firm_maintenance_lapses.py",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

def load_artifact(name: str) -> pd.DataFrame:
    """Read a canonical CSV artifact, or return an empty frame with a hint."""
    path = ARTIFACTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first."
        )
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

## Eligibility funnel

How many matched firms actually carry signal: total patents → checkpoint-eligible →
lapsed. Firms with zero eligible patents contribute nothing either way and must not be
counted as "not dormant".

In [ ]:
lapses = load_artifact("maintenance lapses")
if lapses.empty:
    funnel = pd.DataFrame()
else:
    funnel = pd.DataFrame(
        {
            "firms": [
                len(lapses),
                int((lapses["fee_eligible_patents"] > 0).sum()),
                int(lapses["portfolio_dormant"].astype(bool).sum()),
            ]
        },
        index=["high-confidence matched", "≥1 checkpoint-eligible patent", "flagged portfolio-dormant"],
    )
    print("By dark bucket:")
    display(
        lapses[lapses["portfolio_dormant"].astype(bool)]
        .groupby("bucket").size().rename("dormant_firms").to_frame()
    )
funnel

## Alternative dormancy definitions

Sweep the two threshold parameters (lapse-share and minimum eligible patents) and watch
the flagged count. A conclusion that survives only at exactly (0.8, 2) is a property of
the definition, not of the firms.

In [ ]:
if lapses.empty:
    sweep = pd.DataFrame()
else:
    eligible = lapses[lapses["fee_eligible_patents"] > 0].copy()
    eligible["lapse_share"] = pd.to_numeric(eligible["lapse_share"], errors="coerce")
    rows = []
    for share_threshold in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
        for min_eligible in (1, 2, 3, 5):
            flagged = eligible[
                (eligible["lapse_share"] >= share_threshold)
                & (eligible["fee_eligible_patents"] >= min_eligible)
            ]
            rows.append({
                "lapse_share_threshold": share_threshold,
                "min_eligible_patents": min_eligible,
                "firms_flagged": len(flagged),
            })
    sweep = pd.DataFrame(rows).pivot(
        index="lapse_share_threshold", columns="min_eligible_patents", values="firms_flagged"
    )
sweep

## Boundary sample

Firms just below the canonical flag — high lapse share but under a threshold — are the
ones a definition change would reclassify. Review them deterministically.

In [ ]:
if lapses.empty:
    boundary = pd.DataFrame()
else:
    eligible = lapses[lapses["fee_eligible_patents"] > 0].copy()
    eligible["lapse_share"] = pd.to_numeric(eligible["lapse_share"], errors="coerce")
    near = eligible[
        (~eligible["portfolio_dormant"].astype(bool)) & (eligible["lapse_share"] >= 0.5)
    ]
    boundary = near.sample(min(20, len(near)), random_state=RANDOM_SEED)
boundary

## Interpretation log

| Observation | Definition dependence | Alternative explanation | Defensible statement |
|---|---|---|---|
| _Draft_ | _Sweep cell result_ | _Deliberate abandonment / pivot_ | _Weak negative only_ |